# Public Transport Telemetry - MVP Notebook 00_setup_and_schema: environment setup + table schemas
Safe to rerun. Bronze is append-only; Silver/Gold are overwritten downstream.

In [0]:
# Use UTC for consistent event-time handling
print("session timeZone =", spark.conf.get("spark.sql.session.timeZone"))
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Use a database for the MVP (managed storage, no explicit DBFS paths)
spark.sql("CREATE DATABASE IF NOT EXISTS azure_streaming_mvp")
spark.sql("USE azure_streaming_mvp")

session timeZone = Etc/UTC


DataFrame[]

In [0]:
RESET = True # drop and recreate tables (set it as True)

def reset_tables():
    tables = [
        "bronze_events",
        "silver_transit_metrics",
        "silver_weather_metrics",
        "gold_route_kpi_window",
        "gold_route_kpi_daily",
        "gold_pipeline_metrics_window",
    ]
    for t in tables:
        spark.sql(f"DROP TABLE IF EXISTS `{t}`")
    print("Tables dropped:", ", ".join(tables))

# Clean slate
if RESET:
    reset_tables()

Tables dropped: bronze_events, silver_transit_metrics, silver_weather_metrics, gold_route_kpi_window, gold_route_kpi_daily, gold_pipeline_health_window


Bronze is append-only. Only minimal normalization happens here.

In [0]:
# Create an empty Bronze table with a fixed schema (append-only)
spark.sql("""
CREATE TABLE IF NOT EXISTS bronze_events (
  event_id STRING,
  -- event_time_raw: Raw event time from source (UTC, ISO 8601)
  event_time_raw STRING,
  source STRING,
  entity_type STRING,
  entity_id STRING,
  metric STRING,
  value DOUBLE,
  unit STRING,
  attrs MAP<STRING, STRING>,
  -- event_time_ts: Parsed event time used for windowing
  event_time_ts TIMESTAMP,
  -- ingest_time_ts: Ingestion timestamp (system time)
  ingest_time_ts TIMESTAMP
)
USING DELTA
-- Use Delta Lake (append-only Bronze storage)
""")

DataFrame[]

In [0]:
spark.sql("DESCRIBE TABLE bronze_events").show(truncate=False)

+--------------+------------------+-------+
|col_name      |data_type         |comment|
+--------------+------------------+-------+
|event_id      |string            |NULL   |
|event_time_raw|string            |NULL   |
|source        |string            |NULL   |
|entity_type   |string            |NULL   |
|entity_id     |string            |NULL   |
|metric        |string            |NULL   |
|value         |double            |NULL   |
|unit          |string            |NULL   |
|attrs         |map<string,string>|NULL   |
|event_time_ts |timestamp         |NULL   |
|ingest_time_ts|timestamp         |NULL   |
+--------------+------------------+-------+



In [0]:
# ---- Notebook completion signal ----
dbutils.notebook.exit("OK")